# usage of Neo4J

## Connect

In [10]:
from neo4j import GraphDatabase, basic_auth
import sys

# === 配置信息 ===
# 如果是本地运行脚本，地址通常是 bolt://localhost:7687
# 如果在 Docker 容器内运行此脚本，地址可能是 bolt://host.docker.internal:7687
URI = "bolt://localhost:7687" 
USERNAME = "neo4j"
PASSWORD = "password123"  # 对应你 docker-compose.yml 里的 NEO4J_AUTH

def check_connection():
    driver = None
    try:
        # 1. 建立连接驱动
        print(f"🔌 正在尝试连接到 {URI} ...")
        driver = GraphDatabase.driver(URI, auth=basic_auth(USERNAME, PASSWORD))
        
        # 2. 验证连通性 (这一步会真正发起网络请求)
        driver.verify_connectivity()
        print("✅ 连接成功！(Connectivity verified)")

        # 3. 执行一个简单的测试查询
        # 新版 Neo4j 驱动推荐使用 execute_query API
        query = "RETURN 'Hello, Neo4j is working!' AS message, datetime() AS current_time"
        records, summary, keys = driver.execute_query(query)

        # 4. 打印结果
        for record in records:
            print(f"📩 数据库返回消息: {record['message']}")
            print(f"⏰ 数据库当前时间: {record['current_time']}")
            
        print(f"📊 查询消耗时间: {summary.result_available_after} ms")

    except Exception as e:
        print("\n❌ 连接失败！请检查以下几点：")
        print(f"   错误信息: {e}")
        print("   1. Docker 容器启动了吗？(docker ps 查看)")
        print("   2. 端口 7687 映射了吗？")
        print("   3. 密码对不对？(默认是 password123)")
    finally:
        if driver:
            driver.close()

if __name__ == "__main__":
    check_connection()

🔌 正在尝试连接到 bolt://localhost:7687 ...
✅ 连接成功！(Connectivity verified)
📩 数据库返回消息: Hello, Neo4j is working!
⏰ 数据库当前时间: 2026-02-04T14:39:11.165000000+00:00
📊 查询消耗时间: 4 ms


## Insert 

In [5]:
def insert_data(driver, name, age, city):
    """
    插入数据：创建一个 Person 节点。
    使用 MERGE 避免重复创建（如果存在则更新，不存在则创建）。
    """
    cypher_query = """
    MERGE (p:Person {name: $name})
    SET p.age = $age, p.city = $city
    RETURN elementId(p) as id, p.name as name
    """
    # execute_query 是 Neo4j 5.0+ 推荐的简洁写法，自动处理事务
    records, summary, keys = driver.execute_query(
        cypher_query, 
        name=name, 
        age=age, 
        city=city,
        database_="neo4j" # 默认数据库
    )
    print(f"✅ 成功插入/更新用户: {records[0]['name']}")

In [11]:
# 2. 插入测试数据
# 插入三个用户
AUTH = (USERNAME, PASSWORD)
# 修复核心：定义 main 函数
def main():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        # 1. 验证连接
        try:
            driver.verify_connectivity()
            print("连接 Neo4j 成功！\n")
        except Exception as e:
            print(f"连接失败: {e}")
            return  # <--- 这里的 return 现在合法了，因为它在 main() 函数里

        # 2. 插入测试数据
        insert_data(driver, "Alice", 30, "New York")
        insert_data(driver, "Bob", 25, "London")
        insert_data(driver, "Charlie", 35, "Paris")

# 调用 main 函数
if __name__ == "__main__":
    main()

连接 Neo4j 成功！

✅ 成功插入/更新用户: Alice
✅ 成功插入/更新用户: Bob
✅ 成功插入/更新用户: Charlie


## create relaitonship

In [13]:
def create_relationship(driver, name1, name2):
    """
    插入关系：让 person1 KNOWS person2
    """
    cypher_query = """
    MATCH (p1:Person {name: $name1})
    MATCH (p2:Person {name: $name2})
    MERGE (p1)-[r:KNOWS]->(p2)
    RETURN type(r)
    """
    records, summary, keys = driver.execute_query(
        cypher_query,
        name1=name1,
        name2=name2
    )
    if records:
        print(f"✅ 成功建立关系: {name1} -[KNOWS]-> {name2}")
    else:
        print(f"⚠️ 建立关系失败，可能未找到节点")
        
        

In [ ]:
# 2. 插入测试数据
# 插入三个用户
AUTH = (USERNAME, PASSWORD)
# 修复核心：定义 main 函数
def main():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        # 1. 验证连接
        try:
            driver.verify_connectivity()
            print("连接 Neo4j 成功！\n")
        except Exception as e:
            print(f"连接失败: {e}")
            return  # <--- 这里的 return 现在合法了，因为它在 main() 函数里

        # 3. 建立关系
        create_relationship(driver, "Alice", "Bob")     # Alice 认识 Bob
        create_relationship(driver, "Alice", "Charlie") # Alice 认识 Charlie
   
# 调用 main 函数
if __name__ == "__main__":
    main()

连接 Neo4j 成功！

✅ 成功建立关系: Alice -[KNOWS]-> Bob
✅ 成功建立关系: Alice -[KNOWS]-> Charlie


## query

In [17]:
def query_data(driver, name):
    """
    查询数据：查找某个人的朋友
    """
    cypher_query = """
    MATCH (p:Person {name: $name})-[:KNOWS]->(friend)
    RETURN p.name AS me, friend.name AS friend_name, friend.age AS friend_age
    """
    print(cypher_query)
    records, summary, keys = driver.execute_query(
        cypher_query,
        name=name
    )
    
    print(f"\n🔍 查询结果 ({name} 的朋友):")
    if not records:
        print("   没有找到朋友。")
        return

    for record in records:
        # 使用 record['字段名'] 获取数据
        print(f"   - {record['me']} 认识 {record['friend_name']} (Age: {record['friend_age']})")

In [18]:
# 2. 插入测试数据
# 插入三个用户
AUTH = (USERNAME, PASSWORD)
# 修复核心：定义 main 函数
def main():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        # 1. 验证连接
        try:
            driver.verify_connectivity()
            print("连接 Neo4j 成功！\n")
        except Exception as e:
            print(f"连接失败: {e}")
            return  # <--- 这里的 return 现在合法了，因为它在 main() 函数里

        # 4. 查询数据
        query_data(driver, "Alice")
   
# 调用 main 函数
if __name__ == "__main__":
    main()

连接 Neo4j 成功！


    MATCH (p:Person {name: $name})-[:KNOWS]->(friend)
    RETURN p.name AS me, friend.name AS friend_name, friend.age AS friend_age
    

🔍 查询结果 (Alice 的朋友):
   - Alice 认识 Bob (Age: 25)
   - Alice 认识 Charlie (Age: 35)
